# Workshop 3 Lab — Advanced
### Computation 101 · IQ Biology Bootcamp 2026

You're about to reconstruct a real RNA-seq analysis in Google Colab — a full Linux computer in your
browser. **Run each cell in order** (Shift+Enter). Where you see `...`, replace it with your code before running.

*Background:* **ZFP36L2** is an RNA-binding protein that grabs certain messenger RNAs and marks them for
destruction. Knock the gene out in a mouse (**KO**), and those mRNAs are no longer destroyed — so they go
**up**. Researchers measured this in six tissues with RNA-seq, then used DESeq2 to score every gene. A gene
counts as **up-regulated** when its `log2FoldChange > 1` *and* `padj < 0.05`.

**When a cell turns red — read it, don't panic.** Python puts the real problem on the *last* line:
`KeyError: 'padj'` means there's no column by that name; `NameError: name 'up' is not defined` means you
skipped the cell that made `up`. Read bottom-up, fix that one thing, rerun. Still stuck after a couple
tries? **Bring it to the workshop — that's exactly what it's for.**

In [ ]:
# One-time setup. The lines starting with ! are REAL shell commands running in this
# Colab Linux machine — the same commands you'd type on Fiji. This clones the course
# repo (only if it isn't here yet) and points DATA at the ZFP36L2 dataset inside it.
![ -d iqbio-computation-101-2026 ] || git clone --depth 1 https://github.com/gsstephenson/iqbio-computation-101-2026.git
import pandas as pd, os
DATA = 'iqbio-computation-101-2026/data/zfp36l2'
print('data folder contains:', os.listdir(DATA))

## Your job: name it, then prove it's a direct target
You know one gene — `ENSMUSG00000091694` — is up in all six tissues. What is it, and does ZFP36L2
actually bind it (a *direct* target) or is the effect indirect?

In [ ]:
# Rebuild the six up-sets (same as Intermediate).
tissues = ['Lung', 'Liver', 'BM', 'Spleen', 'Ovary', 'Kidney']
up = {t: set(pd.read_csv(os.path.join(DATA, 'de', f'{t}_DE.csv'), index_col=0)
                 .query('log2FoldChange > 1 and padj < 0.05').index) for t in tissues}
GENE = 'ENSMUSG00000091694'

In [ ]:
# The Gene.name column translates the Ensembl ID into a symbol. Look it up in any table.
lung = pd.read_csv(os.path.join(DATA, 'de', 'Lung_DE.csv'), index_col=0)
symbol = lung.loc[GENE, ...]
print(GENE, '=', symbol)

**Apol11b.** Is it a *direct* ZFP36L2 target? The paper attacks this two ways. Genome-wide first: an
**eCLIP** experiment (in MLTC-1 cells) mapped where ZFP36L2 lands on RNA. `derived/eclip_3utr_genes.txt`
holds the genes with a 3′UTR footprint. Intersect them with the up-regulated genes — and check Apol11b.

In [ ]:
eclip = set(open(os.path.join(DATA, 'derived', 'eclip_3utr_genes.txt')).read().split())
print('Apol11b directly bound in its 3′UTR?', GENE in eclip)
# How many up-regulated genes (any tissue) are also directly bound?
up_any = set().union(*up.values())
print('direct targets:', len(up_any & ...))

**Apol11b isn't in the eCLIP set** (`False`) — and that's not a bug, it's in the paper. The eCLIP ran in
one cell line (MLTC-1), where *Apol11b isn't even expressed*, so it couldn't be captured. In the authors'
words: they confirmed ZFP36L2 binds Apol11b's 3′UTR **by gel-shift assay**, 'but surprisingly this gene
was not captured in our eCLIP data set.' The eCLIP still earns its keep genome-wide — here, **17
up-regulated genes** carry a 3′UTR footprint. For Apol11b itself, the proof is its **ARE motif**:
ZFP36L2 recognizes **AU-rich elements** (core `AUUUA`). Count them in a sequence:

In [ ]:
def count_ares(seq):
    return sum(1 for i in range(len(seq) - 4) if seq[i:i+5] == ...)
print(count_ares('CAUUUAUUUAG'))   # expect 2
print(count_ares('UAUUUAU'))       # Apol11b's ARE

The evidence chain, exactly as the paper builds it: **(1)** Apol11b is the *only* gene up in all six
tissues; **(2)** its 3′UTR carries a single 7-mer ARE (`UAUUUAU`), ZFP36L2's recognition motif; **(3)**
ZFP36L2 **directly binds** the Apol11b probe in a **gel-shift assay** (Fig 5C). The eCLIP maps the binding
mode genome-wide but misses Apol11b — a documented limit, since eCLIP only sees genes expressed in its
cell line. Several methods, each partial, converging on one answer. That's how the paper is built.

## The MA plot — DESeq2's own diagnostic
Before trusting fold changes, DESeq2 looks at an MA plot: mean expression (log scale) across, log2 fold
change up. A sane normalization makes the cloud hug 0. Plot it for bone marrow.

In [ ]:
import matplotlib.pyplot as plt
bm = pd.read_csv(os.path.join(DATA, 'de', 'BM_DE.csv'), index_col=0)
sig = (bm['log2FoldChange'].abs() > 1) & (bm['padj'] < 0.05)
plt.figure(figsize=(6, 4))
plt.scatter(bm['baseMean'], bm['log2FoldChange'], s=3, c='lightgray')
plt.scatter(bm.loc[sig, 'baseMean'], bm.loc[sig, 'log2FoldChange'], s=3, c='crimson')
plt.xscale(...)                       # mean expression spans orders of magnitude
plt.axhline(0, color='k', lw=0.5)
plt.xlabel('mean expression'); plt.ylabel('log2 fold change'); plt.title('Bone marrow — MA plot'); plt.show()

## So what? — functional enrichment
You named Apol11b and tested direct binding. But the up-regulated *set* as a whole — what's it enriched for?
This is the field's daily bread: hand a gene list to an enrichment tool and ask which biological processes are
over-represented. `gseapy` queries **Enrichr** (needs internet — Colab has it).

In [ ]:
!pip install gseapy   # ~30s; if it errors, just re-run this cell
import gseapy as gp
bm = pd.read_csv(os.path.join(DATA, 'de', 'BM_DE.csv'), index_col=0)
up_syms = bm[(bm['log2FoldChange'] > 1) & (bm['padj'] < 0.05)]['Gene.name'].dropna().tolist()
enr = gp.enrichr(gene_list=up_syms, gene_sets=['GO_Biological_Process_2021'], organism='...')   # mouse genes!
print(enr.results.sort_values('Adjusted P-value')[['Term','Adjusted P-value','Overlap']].head(8).to_string(index=False))

The top terms are your **'so what'** — immune / chemokine / inflammatory processes, consistent with ZFP36L2
normally destabilizing inflammatory mRNAs (knock it out → they persist → they read as 'up'). Two pitfalls to
carry with you: (1) the **background set** matters — enrichment is always *relative* to some universe of genes;
(2) enrichment runs its *own* multiple testing — that's exactly what the **Adjusted P-value** column is (padj, again).

## One last lesson: the statistic won't save you
You just drew pictures of your data. Here's *why* that matters — a trick every bioinformatician meets once.
Four tiny datasets. First, the numbers:

In [ ]:
import numpy as np, matplotlib.pyplot as plt
x123 = [10,8,13,9,11,14,6,4,12,7,5]; x4 = [8,8,8,8,8,8,8,19,8,8,8]
quartet = {'I':(x123,[8.04,6.95,7.58,8.81,8.33,9.96,7.24,4.26,10.84,4.82,5.68]),
           'II':(x123,[9.14,8.14,8.74,8.77,9.26,8.10,6.13,3.10,9.13,7.26,4.74]),
           'III':(x123,[7.46,6.77,12.74,7.11,7.81,8.84,6.08,5.39,8.15,6.42,5.73]),
           'IV':(x4,[6.58,5.76,7.71,8.84,8.47,7.04,5.25,12.50,5.56,7.91,6.89])}
for k,(x,y) in quartet.items():
    x=np.array(x,float); y=np.array(y,float); m,b=np.polyfit(x,y,1)
    print(f'{k:3}  mean_x={x.mean():.1f}  mean_y={y.mean():.2f}  corr={np.corrcoef(x,y)[0,1]:.3f}  fit: y={m:.2f}x+{b:.2f}')

Identical — same means, near-identical correlation (≈0.816), the same best-fit line. By the numbers, one dataset.
Now **look**:

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(7, 6))
for ax,(k,(x,y)) in zip(axes.ravel(), quartet.items()):
    x=np.array(x,float); y=np.array(y,float); m,b=np.polyfit(x,y,1)
    ax.scatter(x, y, c='crimson'); ax.plot([2,20],[m*2+b, m*20+b], 'k--', lw=0.8)
    ax.set_title(f'Set {k}'); ax.set_xlim(2,20); ax.set_ylim(2,14)
plt.tight_layout(); plt.show()

Same statistics, four different truths: **I** a real linear trend; **II** an obvious curve (a straight line
is the *wrong model*); **III** a clean line wrecked by one **outlier**; **IV** a single high-leverage point
that invents the whole correlation. This is **Anscombe's Quartet** (1973) — and it is the entire reason you
plot. The number never warned you; you had to look.

It's also why RNA-seq lives in **log2 space** and why DESeq2 normalizes by median-of-ratios instead of raw
ratios: choose the right transform, then *validate by eye* with a volcano or MA plot — exactly what you just
did. Don't hedge the statistic. Look at your data, pick the right normalization, and trust the picture.

## One more thing

The dataset you just analyzed isn't a textbook toy. It's a real, published study —
*The RNA binding protein ZFP36L2 displays tissue-selective mRNA targeting in mice*, **RNA Biology (2026)**.

Its first author is **George** — a first-year in your own program, who a year ago sat exactly where
you're sitting now. Everything you just reproduced, a peer generated. That's the whole arc of this
bootcamp: the person one year ahead of you did real science, and today you re-derived their headline
result from their deposited data. Next year, someone reproduces yours.